# Differentiable Gaussian Rasterization

## 목차
1. [개요](#개요)
2. [diff-gaussian-rasterization](#diff-gaussian-rasterization)
   - [필요성](#필요성)
   - [핵심 아이디어](#핵심-아이디어)
   - [Forward Pass 구현](#forward-pass-구현)
   - [Backward Pass 구현](#backward-pass-구현)
   - [최적화 기법](#최적화-기법)
3. [설치 및 사용](#설치-및-사용)
4. [참고자료](#참고자료)

---

## 개요

**diff-gaussian-rasterization**은 3D Gaussian들을 2D 이미지로 렌더링하는 커스텀 CUDA 레이어입니다. 3D Gaussian Splatting의 핵심 구성 요소로, 역전파(backpropagation)를 지원하여 end-to-end 학습이 가능합니다.

---

## diff-gaussian-rasterization

### 필요성

3D Gaussian Splatting은 수백만 개의 3D Gaussian을 실시간으로 렌더링해야 합니다:
- **일반 PyTorch**: 너무 느림 (Python overhead, 비효율적인 메모리 접근)
- **기존 렌더러**: 2D projection과 alpha blending에 최적화되지 않음
- **커스텀 CUDA 구현**: 타일 기반 병렬 처리로 10-100배 빠른 속도

### 핵심 아이디어

#### 1. 3D Gaussian 표현

각 Gaussian $G_i$는 다음 파라미터로 정의됩니다:

**위치 (Position)**:
$$\boldsymbol{\mu}_i \in \mathbb{R}^3$$

**색상 (Color, Spherical Harmonics)**:
$$\text{SH coefficients: } c_{i,\ell}^m \text{ for } \ell \in [0, 3], m \in [-\ell, \ell]$$

**불투명도 (Opacity)**:
$$o_i \in [0, 1]$$

**회전 (Rotation, Quaternion)**:
$$\boldsymbol{q}_i = (q_w, q_x, q_y, q_z) \text{ with } \|\boldsymbol{q}_i\| = 1$$

**스케일 (Scale)**:
$$\boldsymbol{s}_i = (s_x, s_y, s_z) \in \mathbb{R}^3_+$$

**3D 공분산 행렬**:
$$\boldsymbol{\Sigma}_i = \boldsymbol{R}_i \boldsymbol{S}_i \boldsymbol{S}_i^T \boldsymbol{R}_i^T$$

여기서 $\boldsymbol{R}_i$는 quaternion $\boldsymbol{q}_i$로부터 계산된 회전 행렬입니다.

#### 2. 2D Projection

3D Gaussian을 카메라 평면에 투영:

**Projection 변환**:
$$\boldsymbol{\mu}'_i = \boldsymbol{W} \boldsymbol{\mu}_i$$

여기서 $\boldsymbol{W}$는 world-to-view 변환 행렬입니다.

**2D 공분산 계산 (EWA Splatting)**:
$$\boldsymbol{\Sigma}'_i = \boldsymbol{J} \boldsymbol{W} \boldsymbol{\Sigma}_i \boldsymbol{W}^T \boldsymbol{J}^T$$

여기서 $\boldsymbol{J}$는 perspective projection의 Jacobian 행렬:
$$\boldsymbol{J} = \begin{bmatrix}
\frac{f_x}{z} & 0 & -\frac{f_x x}{z^2} \\
0 & \frac{f_y}{z} & -\frac{f_y y}{z^2}
\end{bmatrix}$$

**2D Gaussian 형태**:
$$G'_i(\boldsymbol{x}) = e^{-\frac{1}{2}(\boldsymbol{x} - \boldsymbol{\mu}'_i)^T (\boldsymbol{\Sigma}'_i)^{-1} (\boldsymbol{x} - \boldsymbol{\mu}'_i)}$$

**Covariance Regularization과 Antialiasing**:

원본 구현에서는 2D covariance 대각 성분에 상수 $h$를 추가합니다:
$$\boldsymbol{\Sigma}'_{\text{regularized}} = \boldsymbol{\Sigma}' + h \mathbf{I}$$

여기서 $h = 0.3$ (픽셀 단위). 이 연산은 **두 가지 목적**을 가지며, 각각 다르게 적용됩니다:

**1. 수치적 안정성 (항상 적용)**

h_var 추가는 `antialiasing` 플래그와 관계없이 **항상 수행**됩니다:

In [ ]:
// 이 부분은 조건문 밖에 있음 - 항상 실행됨
cov.x += h_var;  // σ_xx → σ_xx + h
cov.z += h_var;  // σ_yy → σ_yy + h



**왜 항상 필요한가?**

- **역행렬 계산 보장**: Covariance 행렬의 determinant가 0에 가까우면 `det_inv = 1/det`이 발산
- **최소 분산 보장**: 매우 작은 Gaussian은 eigenvalue ≈ 0이 되어 수치적으로 불안정
- **Gradient 안정화**: 너무 날카로운 Gaussian은 gradient가 급격히 변하여 학습 불안정
- $h$ 추가로 eigenvalue ≥ $h$를 보장 → 항상 invertible한 covariance

**2. Opacity 보정 (antialiasing=true일 때만 적용)**

In [ ]:
float h_convolution_scaling = 1.0f;  // 기본값: 보정 없음
if(antialiasing)
    h_convolution_scaling = sqrt(max(0.000025f, det_cov / det_cov_plus_h_cov));



h_var를 추가하면 Gaussian이 커지므로, **같은 "질량"을 유지**하려면 opacity를 줄여야 합니다:
- `antialiasing=false`: h_var 추가로 Gaussian이 커지지만 opacity 보정 없음
- `antialiasing=true`: Gaussian이 커진 만큼 opacity를 보정하여 총 기여도 유지

**요약**:
| 연산 | 목적 | 적용 조건 |
|------|------|----------|
| `cov += h*I` | 수치적 안정성 | 항상 |
| `opacity *= scaling` | 질량 보존 (antialiasing) | `antialiasing=true`일 때만 |

**computeCov2D 함수 - EWA Splatting 구현** (forward.cu Lines 78-116):

In [ ]:
// Forward version of 2D covariance matrix computation
__device__ float3 computeCov2D(const float3& mean, float focal_x, float focal_y, 
                                float tan_fovx, float tan_fovy, 
                                const float* cov3D, const float* viewmatrix)
{
    // EWA Splatting (Zwicker et al., 2002) equations 29 & 31
    // Transform to view space
    float3 t = transformPoint4x3(mean, viewmatrix);
    
    // Clamp to viewport bounds (frustum culling for gradients)
    const float limx = 1.3f * tan_fovx;
    const float limy = 1.3f * tan_fovy;
    const float txtz = t.x / t.z;
    const float tytz = t.y / t.z;
    t.x = min(limx, max(-limx, txtz)) * t.z;
    t.y = min(limy, max(-limy, tytz)) * t.z;
    
    // Jacobian of perspective projection
    glm::mat3 J = glm::mat3(
        focal_x / t.z, 0.0f, -(focal_x * t.x) / (t.z * t.z),
        0.0f, focal_y / t.z, -(focal_y * t.y) / (t.z * t.z),
        0, 0, 0);
    
    // View transformation matrix (world → view)
    glm::mat3 W = glm::mat3(
        viewmatrix[0], viewmatrix[4], viewmatrix[8],
        viewmatrix[1], viewmatrix[5], viewmatrix[9],
        viewmatrix[2], viewmatrix[6], viewmatrix[10]);
    
    glm::mat3 T = W * J;
    
    // 3D covariance matrix (symmetric, upper triangular stored)
    glm::mat3 Vrk = glm::mat3(
        cov3D[0], cov3D[1], cov3D[2],
        cov3D[1], cov3D[3], cov3D[4],
        cov3D[2], cov3D[4], cov3D[5]);
    
    // 2D covariance: Σ' = J * W * Σ * W^T * J^T
    glm::mat3 cov = glm::transpose(T) * glm::transpose(Vrk) * T;
    
    // Return upper triangular: [σ_xx, σ_xy, σ_yy]
    return { float(cov[0][0]), float(cov[0][1]), float(cov[1][1]) };
}



**Forward Pass 구현** (forward.cu Lines 220-274):

In [ ]:
// Compute 2D screen-space covariance matrix
// computeCov2D returns float3 = [σ_xx, σ_xy, σ_yy] (upper triangular)
float3 cov = computeCov2D(p_orig, focal_x, focal_y, tan_fovx, tan_fovy, cov3D, viewmatrix);

// === Covariance Regularization ===
constexpr float h_var = 0.3f;

// 원본 covariance의 determinant: det(Σ') = σ_xx·σ_yy - σ_xy²
// (antialiasing 보정을 위해 미리 계산)
const float det_cov = cov.x * cov.z - cov.y * cov.y;

// 수치적 안정성을 위해 대각 성분에 h 추가 (항상 실행됨!)
// Σ'_regularized = [[σ_xx+h, σ_xy], [σ_xy, σ_yy+h]]
cov.x += h_var;  // σ_xx → σ_xx + h
cov.z += h_var;  // σ_yy → σ_yy + h
// 이로써 eigenvalue ≥ h 보장 → 역행렬 계산 안정성 확보

// regularization 후 determinant: det(Σ' + hI)
const float det_cov_plus_h_cov = cov.x * cov.z - cov.y * cov.y;

// === Opacity 보정 (antialiasing=true일 때만) ===
// Gaussian이 커진 만큼 opacity를 줄여 총 기여도(질량) 보존
float h_convolution_scaling = 1.0f;  // 기본값: 보정 없음
if(antialiasing)
    h_convolution_scaling = sqrt(max(0.000025f, det_cov / det_cov_plus_h_cov));
    // sqrt(원본면적 / 확장면적) → 면적이 커지면 opacity 감소

// === Invert covariance (conic matrix) ===
// EWA algorithm: Σ^-1
const float det = det_cov_plus_h_cov;
if (det == 0.0f)
    return;

float det_inv = 1.f / det;
float3 conic = { cov.z * det_inv, -cov.y * det_inv, cov.x * det_inv };

// Compute extent (bounding box radius from eigenvalues)
float mid = 0.5f * (cov.x + cov.z);
float lambda1 = mid + sqrt(max(0.1f, mid * mid - det));
float lambda2 = mid - sqrt(max(0.1f, mid * mid - det));
float my_radius = ceil(3.f * sqrt(max(lambda1, lambda2)));  // 3σ rule

// ... (tile allocation, color computation)

// Store opacity with antialiasing correction
float opacity = opacities[idx];
conic_opacity[idx] = { conic.x, conic.y, conic.z, opacity * h_convolution_scaling };



**Backward Pass 구현** (backward.cu Lines 210-244):

In [ ]:
constexpr float h_var = 0.3f;
float d_inside_root = 0.f;

if(antialiasing)
{
    // Forward와 동일하게 재계산
    const float det_cov = c_xx * c_yy - c_xy * c_xy;
    c_xx += h_var;
    c_yy += h_var;
    const float det_cov_plus_h_cov = c_xx * c_yy - c_xy * c_xy;
    const float h_convolution_scaling = sqrt(max(0.000025f, det_cov / det_cov_plus_h_cov));
    
    // Opacity gradient 역전파
    const float dL_dopacity_v = dL_dopacity[idx];
    const float d_h_convolution_scaling = dL_dopacity_v * opacities[idx];
    dL_dopacity[idx] = dL_dopacity_v * h_convolution_scaling;
    
    // sqrt 내부에 대한 gradient (chain rule)
    d_inside_root = (det_cov / det_cov_plus_h_cov) <= 0.000025f ? 0.f 
                    : d_h_convolution_scaling / (2 * h_convolution_scaling);
}

// Scaling factor gradient를 covariance gradient로 전파
if(antialiasing)
{
    // Wolfram Alpha로 계산된 analytic gradient
    // d(det_cov / det_cov_plus_h) / d(c_xx), d(c_yy), d(c_xy)
    const float x = c_xx;
    const float y = c_yy;
    const float z = c_xy;
    const float w = h_var;
    const float denom_f = d_inside_root / sq(w * w + w * (x + y) + x * y - z * z);
    
    const float dL_dx = w * (w * y + y * y + z * z) * denom_f;
    const float dL_dy = w * (w * x + x * x + z * z) * denom_f;
    const float dL_dz = -2.f * w * z * (w + x + y) * denom_f;
    
    dL_dc_xx = dL_dx;
    dL_dc_yy = dL_dy;
    dL_dc_xy = dL_dz;
}



**수학적 세부사항**:

Scaling factor의 gradient:

$$\frac{\partial}{\partial \sigma_{xx}} \sqrt{\frac{\det(\Sigma)}{\det(\Sigma + hI)}}$$

여기서:
- $\det(\Sigma) = \sigma_{xx}\sigma_{yy} - \sigma_{xy}^2$
- $\det(\Sigma + hI) = (\sigma_{xx}+h)(\sigma_{yy}+h) - \sigma_{xy}^2$

Chain rule과 quotient rule 적용:

$$\frac{\partial}{\partial \sigma_{xx}} = \frac{1}{2\sqrt{\text{ratio}}} \cdot \frac{\partial \text{ratio}}{\partial \sigma_{xx}}$$

Wolfram Alpha를 사용하여 analytic gradient 계산 (주석의 링크 참조)

#### 3. Tile-Based Rasterization

**화면 분할**:
- 이미지를 16×16 픽셀 타일로 분할
- 각 타일을 독립적인 CUDA 블록으로 처리

**타일-Gaussian 할당**:

```
for each Gaussian G_i:
    if G_i overlaps tile T:
        add G_i to T's list
```



**정렬**:
- 각 타일 내에서 Gaussian들을 깊이(depth) 순으로 정렬
- front-to-back 순서로 alpha blending 수행

#### 4. Alpha Blending (Volume Rendering)

픽셀 $\boldsymbol{x}$의 최종 색상:

$$C(\boldsymbol{x}) = \sum_{i \in \mathcal{N}} c_i \cdot \alpha_i \cdot T_i$$

여기서:
- $c_i$: Gaussian $i$의 색상 (Spherical Harmonics로부터 계산)
- $\alpha_i = o_i \cdot G'_i(\boldsymbol{x})$: Gaussian의 알파 값
- $T_i = \prod_{j=1}^{i-1}(1 - \alpha_j)$: 투과율 (transmittance)

**조기 종료 (Early Stopping)**:
$$\text{if } T_i < \epsilon \text{ (예: 0.0001), stop}$$

대부분의 빛이 차단되면 뒤의 Gaussian들은 무시해도 됩니다.

### Forward Pass 구현

> **실제 구현 코드**: [cuda_rasterizer/forward.cu](../../references/gaussian-splatting/submodules/diff-gaussian-rasterization/cuda_rasterizer/forward.cu)

#### CUDA 커널 구조

**주요 함수 및 위치**:
- `preprocessCUDA` (Lines 177-277): 3D → 2D projection, covariance 계산, frustum culling
- `renderCUDA` (Lines 283-412): Tile-based rasterization, alpha blending
- `computeCov3D` (Lines 126-158): Scale & rotation → 3D covariance matrix
- `computeCov2D` (Lines 82-124): 3D → 2D covariance projection (EWA Splatting)
- `computeColorFromSH` (Lines 23-80): Spherical harmonics → RGB

In [ ]:
// 설정 (config.h Lines 10-11)
#define BLOCK_X 16
#define BLOCK_Y 16
#define BLOCK_SIZE (BLOCK_X * BLOCK_Y)  // 256

// Forward 커널 (Lines 283-412)
template <uint32_t CHANNELS>
__global__ void __launch_bounds__(BLOCK_X * BLOCK_Y)
renderCUDA(
    const uint2* __restrict__ ranges,           // tile-Gaussian ranges
    const uint32_t* __restrict__ point_list,    // sorted Gaussian IDs
    int W, int H,
    const float2* __restrict__ points_xy_image, // [N, 2] 2D positions
    const float* __restrict__ features,         // [N, C] colors/SH
    const float4* __restrict__ conic_opacity,   // [N, 4] (conic + opacity)
    const float* __restrict__ bg_color,         // [3]
    float* __restrict__ out_color,              // [H, W, C]
    const float* __restrict__ depths,           // [N, 1]
    float* __restrict__ invdepth                // [H, W, 1]
)



#### 단계별 처리

**1. Preprocessing** (Lines 177-277 in `preprocessCUDA`):

In [ ]:
// 각 Gaussian에 대해 병렬 처리
__global__ void preprocessCUDA(...) {
    auto idx = cg::this_grid().thread_rank();
    if (idx >= P) return;
    
    // Initialize
    radii[idx] = 0;
    tiles_touched[idx] = 0;
    
    // Frustum culling (Line 206)
    float3 p_view;
    if (!in_frustum(idx, orig_points, viewmatrix, projmatrix, prefiltered, p_view))
        return;
    
    // 3D → 2D projection (Lines 209-210)
    float3 p_orig = { orig_points[3 * idx], ... };
    float3 p_view = transformPoint4x3(p_orig, viewmatrix);
    float3 p_proj = transformPoint4x3(p_orig, projmatrix);
    
    // 3D covariance 계산 (Lines 215-217)
    float3x3 cov3D;
    computeCov3D(scales[idx], scale_modifier, rotations[idx], cov3D);
    
    // 2D covariance 계산 (Lines 220-224)
    float3 cov2D = computeCov2D(p_orig, focal_x, focal_y, tan_fovx, tan_fovy, 
                                 cov3D, viewmatrix);
    
    // Covariance regularization (Lines 226-231)
    // 수치적 안정성을 위해 항상 적용, antialiasing일 때만 opacity 보정
    constexpr float h_var = 0.3f;
    float det_cov = cov2D.x * cov2D.z - cov2D.y * cov2D.y;
    cov2D.x += h_var;  // 항상 적용: 수치적 안정성
    cov2D.z += h_var;
    float det_cov_plus_h = cov2D.x * cov2D.z - cov2D.y * cov2D.y;
    // antialiasing=true일 때만 opacity 보정
    float h_convolution_scaling = sqrt(max(0.000025f, det_cov / det_cov_plus_h));
    
    // Bounding box & radius 계산 (Lines 243-247)
    float det = (cov2D.x * cov2D.z - cov2D.y * cov2D.y);
    float mid = 0.5f * (cov2D.x + cov2D.z);
    float lambda1 = mid + sqrt(max(0.1f, mid * mid - det));
    float lambda2 = mid - sqrt(max(0.1f, mid * mid - det));
    float my_radius = ceil(3.f * sqrt(max(lambda1, lambda2)));
    
    // 2D screen position (Lines 234-236)
    float2 point_image = { ndc2Pix(p_proj.x, W), ndc2Pix(p_proj.y, H) };
    
    // Tile range 계산 (Lines 242-251)
    uint2 rect_min, rect_max;
    getRect(point_image, my_radius, rect_min, rect_max, grid);
    
    // Color 계산: SH → RGB (Lines 253-259)
    glm::vec3 result = computeColorFromSH(idx, deg, max_coeffs, means, cam_pos, 
                                          dc, shs, clamped);
    
    // Store results
    radii[idx] = my_radius;
    points_xy_image[idx] = point_image;
    depths[idx] = p_view.z;
    rgb[idx * C + ...] = result;
    conic_opacity[idx] = { conic.x, conic.y, conic.z, opacities[idx] * h_convolution_scaling };
    tiles_touched[idx] = (rect_max.y - rect_min.y) * (rect_max.x - rect_min.x);
}



**2. 타일별 정렬**

이 단계는 CPU/GPU에서 CUB (CUDA Unbound) 라이브러리를 사용하여 수행됩니다.

In [ ]:
// Sort Gaussians by (tile_id, depth) using CUB radix sort
cub::DeviceRadixSort::SortPairs(...)



**3. Rendering** (Lines 283-412 in `renderCUDA`):

In [ ]:
template <uint32_t CHANNELS>
__global__ void __launch_bounds__(BLOCK_X * BLOCK_Y)
renderCUDA(...) {
    // Tile & pixel identification (Lines 302-310)
    auto block = cg::this_thread_block();
    uint32_t horizontal_blocks = (W + BLOCK_X - 1) / BLOCK_X;
    uint2 pix_min = { block.group_index().x * BLOCK_X, block.group_index().y * BLOCK_Y };
    uint2 pix = { pix_min.x + block.thread_index().x, pix_min.y + block.thread_index().y };
    uint32_t pix_id = W * pix.y + pix.x;
    float2 pixf = { (float)pix.x, (float)pix.y };
    
    bool inside = pix.x < W && pix.y < H;
    bool done = !inside;
    
    // Load tile range (Lines 315-317)
    uint32_t tile_id = block.group_index().y * horizontal_blocks + block.group_index().x;
    uint2 range = ranges[tile_id];
    int rounds = ((range.y - range.x + BLOCK_SIZE - 1) / BLOCK_SIZE);
    
    // Shared memory for cooperative loading (Lines 331-333)
    __shared__ int collected_id[BLOCK_SIZE];
    __shared__ float2 collected_xy[BLOCK_SIZE];
    __shared__ float4 collected_conic_opacity[BLOCK_SIZE];
    
    // Initialize accumulators (Lines 335-339)
    float T = 1.0f;              // transmittance
    uint32_t contributor = 0;
    float C[CHANNELS] = { 0 };   // accumulated color
    float expected_invdepth = 0.0f;
    
    // Iterate over Gaussian batches (Lines 341-410)
    for (int i = 0; i < rounds; i++, toDo -= BLOCK_SIZE) {
        // Early termination check (Lines 344-346)
        int num_done = __syncthreads_count(done);
        if (num_done == BLOCK_SIZE)
            break;
        
        // Collectively fetch Gaussian data (Lines 348-363)
        int progress = i * BLOCK_SIZE + block.thread_rank();
        if (range.x + progress < range.y) {
            int coll_id = point_list[range.x + progress];
            collected_id[block.thread_rank()] = coll_id;
            collected_xy[block.thread_rank()] = points_xy_image[coll_id];
            collected_conic_opacity[block.thread_rank()] = conic_opacity[coll_id];
        }
        block.sync();
        
        // Process Gaussians in this batch (Lines 365-407)
        for (int j = 0; !done && j < min(BLOCK_SIZE, toDo); j++) {
            // Sample T and accumulated color every 32nd Gaussian (bucket)
            if (j % 32 == 0) {
                sampled_T[bucket_idx] = T;
                for (int ch = 0; ch < CHANNELS; ++ch)
                    sampled_ar[bucket_idx * CHANNELS + ch] = C[ch];
                sampled_ard[bucket_idx] = expected_invdepth;
                ++bucket_idx;
            }
            
            contributor++;
            
            // 2D Gaussian evaluation (Lines 369-377)
            int gaussian_id = collected_id[j];
            float2 xy = collected_xy[j];
            float2 d = { xy.x - pixf.x, xy.y - pixf.y };
            float4 con_o = collected_conic_opacity[j];
            
            float power = -0.5f * (con_o.x * d.x * d.x + con_o.z * d.y * d.y) 
                          - con_o.y * d.x * d.y;
            if (power > 0.0f) continue;
            
            float alpha = min(0.99f, con_o.w * exp(power));
            if (alpha < 1.0f / 255.0f) continue;
            
            // Alpha blending (Lines 390-397)
            float test_T = T * (1.f - alpha);
            if (test_T < 0.0001f) {
                done = true;
                continue;
            }
            
            // Accumulate color (Lines 382-384)
            for (int ch = 0; ch < CHANNELS; ch++)
                C[ch] += features[gaussian_id * CHANNELS + ch] * alpha * T;
            
            // Accumulate depth (Lines 399-400)
            float invdepth_contrib = 1.0f / depths[gaussian_id];
            expected_invdepth += invdepth_contrib * alpha * T;
            
            T = test_T;
            last_contributor = contributor;
        }
    }
    
    // Write output (Lines 413-422)
    if (inside) {
        final_T[pix_id] = T;
        n_contrib[pix_id] = last_contributor;
        for (int ch = 0; ch < CHANNELS; ch++)
            out_color[ch * H * W + pix_id] = C[ch] + T * bg_color[ch];
        invdepth[pix_id] = expected_invdepth;
    }
}



### Backward Pass 구현

> **실제 구현 코드**: [cuda_rasterizer/backward.cu](../../references/gaussian-splatting/submodules/diff-gaussian-rasterization/cuda_rasterizer/backward.cu)

**주요 함수 및 위치**:
- `PerGaussianRenderCUDA` (Lines 453-669): Tile-based backward pass, gradient 누적
- `preprocessCUDA` (Lines 397-449): Projection backward pass
- `computeCov2DCUDA` (Lines 149-395): 2D covariance gradient
- `computeColorFromSH` (Lines 22-147): Spherical harmonics gradient

역전파의 핵심: **chain rule을 역순으로 적용**

#### 기본 수식

**최종 손실**:
$$L = \text{Loss}(C(\boldsymbol{x}), C_{\text{GT}}(\boldsymbol{x}))$$

**색상에 대한 gradient** (forward pass에서 계산됨):
$$\frac{\partial L}{\partial C(\boldsymbol{x})}$$

#### 단계별 Gradient 계산

**1. Alpha Blending Gradient (Backward Accumulation)**

Forward pass에서 각 픽셀마다 다음을 저장합니다:
- `final_T`: 최종 투과율
- `n_contrib`: 기여한 Gaussian 개수 (early stopping 지점)

Backward pass는 **역순**으로 진행됩니다 (last contributor → first):

```
T = final_T  // 저장된 최종 투과율로 시작
for i = n_contrib down to 1:
    dL_dalpha_i = dL_dC * c_i * T + (다음 Gaussian들의 누적 효과)
    T = T / (1 - alpha_i)  // 이전 상태로 복원
```



수식으로 표현:
$$\frac{\partial L}{\partial c_i} = \frac{\partial L}{\partial C} \cdot \alpha_i \cdot T_i$$

$$\frac{\partial L}{\partial \alpha_i} = \frac{\partial L}{\partial C} \cdot \left(c_i \cdot T_i - \sum_{j>i}^{N} c_j \alpha_j T_j / (1 - \alpha_i)\right)$$

**2. 2D Gaussian Gradient**

$$\frac{\partial L}{\partial o_i} = \frac{\partial L}{\partial \alpha_i} \cdot G'_i(\boldsymbol{x})$$

$$\frac{\partial L}{\partial \boldsymbol{\mu}'_i} = \frac{\partial L}{\partial \alpha_i} \cdot o_i \cdot \frac{\partial G'_i}{\partial \boldsymbol{\mu}'_i}$$

$$\frac{\partial L}{\partial \boldsymbol{\Sigma}'_i} = \frac{\partial L}{\partial \alpha_i} \cdot o_i \cdot \frac{\partial G'_i}{\partial \boldsymbol{\Sigma}'_i}$$

**3. Projection Gradient (3D → 2D)**

$$\frac{\partial L}{\partial \boldsymbol{\mu}_i} = \frac{\partial L}{\partial \boldsymbol{\mu}'_i} \cdot \frac{\partial \boldsymbol{\mu}'_i}{\partial \boldsymbol{\mu}_i}$$

$$\frac{\partial L}{\partial \boldsymbol{\Sigma}_i} = \frac{\partial L}{\partial \boldsymbol{\Sigma}'_i} \cdot \frac{\partial \boldsymbol{\Sigma}'_i}{\partial \boldsymbol{\Sigma}_i}$$

**4. Quaternion & Scale Gradient**

$$\frac{\partial L}{\partial \boldsymbol{q}_i} = \frac{\partial L}{\partial \boldsymbol{\Sigma}_i} \cdot \frac{\partial \boldsymbol{\Sigma}_i}{\partial \boldsymbol{R}_i} \cdot \frac{\partial \boldsymbol{R}_i}{\partial \boldsymbol{q}_i}$$

$$\frac{\partial L}{\partial \boldsymbol{s}_i} = \frac{\partial L}{\partial \boldsymbol{\Sigma}_i} \cdot \frac{\partial \boldsymbol{\Sigma}_i}{\partial \boldsymbol{S}_i}$$

**5. Spherical Harmonics Gradient**

> **함수**: `computeColorFromSH` (Lines 22-147 in backward.cu)

SH gradient 계산은 두 가지 경로로 이루어집니다:

**A. SH 계수에 대한 gradient** (직접):

각 SH 계수 $c_{l,m}$에 대한 gradient는 해당 basis function 값을 곱하여 계산됩니다:

$$\frac{\partial L}{\partial c_{l,m}} = Y_l^m(\mathbf{d}) \cdot \frac{\partial L}{\partial \text{RGB}}$$

**B. 위치에 대한 gradient** (viewing direction을 통해 간접):

색상은 normalized viewing direction $\mathbf{d} = \frac{\mathbf{p} - \mathbf{c}}{||\mathbf{p} - \mathbf{c}||}$에 의존하므로:

$$\frac{\partial L}{\partial \mathbf{p}} = \frac{\partial L}{\partial \text{RGB}} \cdot \frac{\partial \text{RGB}}{\partial \mathbf{d}} \cdot \frac{\partial \mathbf{d}}{\partial \mathbf{p}}$$

이 계산에는 normalized vector의 gradient (chain rule)와 forward에서 저장된 clamping 정보가 사용됩니다.

#### CUDA Backward 커널

> **함수**: `PerGaussianRenderCUDA` (Lines 453-669)

**핵심 구조**: Gaussian-centric backward pass (forward는 pixel-centric)

In [ ]:
template<uint32_t C>
__global__ void PerGaussianRenderCUDA(
    const uint2* __restrict__ ranges,
    const uint32_t* __restrict__ point_list,
    int W, int H, int B,  // B = total number of buckets
    const uint32_t* __restrict__ per_tile_bucket_offset,
    const uint32_t* __restrict__ bucket_to_tile,
    const float* __restrict__ sampled_T,        // saved T values at bucket boundaries
    const float* __restrict__ sampled_ar,       // saved accumulated colors
    const float* __restrict__ sampled_ard,      // saved accumulated depths
    const float* __restrict__ bg_color,
    const float2* __restrict__ points_xy_image,
    const float4* __restrict__ conic_opacity,
    const float* __restrict__ colors,
    const float* __restrict__ depths,
    const float* __restrict__ final_Ts,         // saved from forward
    const uint32_t* __restrict__ n_contrib,     // saved from forward
    const float* __restrict__ dL_dpixels,       // gradient from loss
    const float* __restrict__ dL_invdepths,     // gradient from depth loss
    float3* __restrict__ dL_dmean2D,            // output gradients
    float4* __restrict__ dL_dconic2D,
    float* __restrict__ dL_dopacity,
    float* __restrict__ dL_dcolors,
    float* __restrict__ dL_dinvdepths
) {
    // Warp-based processing (Lines 477-490)
    auto block = cg::this_thread_block();
    auto my_warp = cg::tiled_partition<32>(block);
    uint32_t global_bucket_idx = block.group_index().x * my_warp.meta_group_size() 
                                 + my_warp.meta_group_rank();
    
    // Each warp processes one "bucket" of 32 consecutive pixels in a tile
    uint32_t tile_id = bucket_to_tile[global_bucket_idx];
    uint2 range = ranges[tile_id];
    
    // Identify this warp's Gaussian (Lines 492-496)
    int splat_idx_in_tile = bucket_idx_in_tile * 32 + my_warp.thread_rank();
    int splat_idx_global = range.x + splat_idx_in_tile;
    int gaussian_id = point_list[splat_idx_global];
    
    // Shared memory for tile pixels (Lines 502-523)
    __shared__ float2 s_pix_xy[THREADS];
    __shared__ float s_T[THREADS];
    __shared__ uint32_t s_nc[THREADS];
    __shared__ float s_dL_dpix[C][THREADS];
    
    // Load pixel data cooperatively (Lines 526-548)
    // ... (cooperative loading of 256 pixels)
    
    // Compute gradients for this Gaussian (Lines 550-646)
    float dL_G = 0.0f;       // gradient w.r.t. Gaussian value
    float dL_A = 0.0f;       // gradient w.r.t. alpha
    float2 dL_dxy = {0, 0};  // gradient w.r.t. 2D position
    float3 dL_dconic = {0, 0, 0};  // gradient w.r.t. conic params
    float dL_dcolor[C] = {0};      // gradient w.r.t. colors
    
    // Iterate through pixels in this tile (Lines 550-630)
    for (int s = 0; s < num_samples_in_bucket; ++s) {
        int thread_data = my_warp.thread_rank() * num_samples + s;
        
        float2 xy = s_pix_xy[thread_data];
        float T = s_T[thread_data];
        
        // 2D Gaussian evaluation (Lines 563-572)
        float2 d = { xy.x - point_xy.x, xy.y - point_xy.y };
        float power = -0.5f * (con_o.x * d.x * d.x + con_o.z * d.y * d.y)
                      - con_o.y * d.x * d.y;
        if (power > 0.0f) continue;
        
        float G = exp(power);
        float alpha = min(0.99f, con_o.w * G);
        if (alpha < 1.0f / 255.0f) continue;
        
        // Gradient from pixel color (Lines 579-589)
        float dL_dTalpha = 0.0f;
        for (int ch = 0; ch < C; ch++) {
            float c = colors[gaussian_id * C + ch];
            float dL_dchan = s_dL_dpix[ch][thread_data];
            dL_dcolor[ch] += dL_dchan * T * alpha;
            dL_dTalpha += dL_dchan * c;
        }
        
        // Gradient from depth (Lines 591-600)
        float invdepth = 1.0f / depths[gaussian_id];
        float dL_dinv = s_dL_dinv[thread_data];
        dL_dinvdepths[gaussian_id] += dL_dinv * T * alpha;
        dL_dTalpha += dL_dinv * invdepth;
        
        // Accumulate α and T*α gradients (Lines 602-612)
        float dL_dalpha = dL_dTalpha * T;
        dL_A += dL_dalpha;
        dL_G += dL_dalpha * con_o.w;
        
        // Gradient w.r.t. 2D position and conic (Lines 614-625)
        float dG_dx = -G * (con_o.x * d.x + con_o.y * d.y);
        float dG_dy = -G * (con_o.y * d.x + con_o.z * d.y);
        dL_dxy.x += dL_G * dG_dx;
        dL_dxy.y += dL_G * dG_dy;
        
        dL_dconic.x += -0.5f * dL_G * G * d.x * d.x;
        dL_dconic.y += -1.0f * dL_G * G * d.x * d.y;
        dL_dconic.z += -0.5f * dL_G * G * d.y * d.y;
    }
    
    // Atomic accumulation (Lines 632-646)
    atomicAdd(&dL_dmean2D[gaussian_id].x, dL_dxy.x);
    atomicAdd(&dL_dmean2D[gaussian_id].y, dL_dxy.y);
    atomicAdd(&dL_dconic2D[gaussian_id].x, dL_dconic.x);
    atomicAdd(&dL_dconic2D[gaussian_id].y, dL_dconic.y);
    atomicAdd(&dL_dconic2D[gaussian_id].z, dL_dconic.z);
    atomicAdd(&dL_dopacity[gaussian_id], dL_A);
    for (int ch = 0; ch < C; ch++)
        atomicAdd(&dL_dcolors[gaussian_id * C + ch], dL_dcolor[ch]);
}



**핵심 차이점**:
- Forward: Pixel마다 스레드 할당, Gaussian 순회, 32개마다 상태 저장 (bucket)
- Backward: Gaussian마다 warp 할당, 저장된 bucket에서 상태 복원하여 영향받는 픽셀 순회
- Bucket: 32개 Gaussian마다 T, accumulated color, depth 저장 → backward에서 재계산 최소화
- 이유: Atomic operation 최소화 + 정확한 transmittance 복원

**Atomic Operations가 필요한 이유**:
- 한 Gaussian이 여러 픽셀에 영향을 주므로, backward에서 여러 픽셀의 gradient가 같은 Gaussian 파라미터에 누적됨
- `atomicAdd` 없이는 race condition 발생 → gradient 손실
- 성능 저하가 있지만 정확한 gradient 계산을 위해 필수

### 최적화 기법

#### 1. Tile-Based Processing
- **병렬성**: 각 타일(16×16 픽셀)이 독립적으로 CUDA 블록으로 처리됨
- **메모리 지역성**: 타일 내 Gaussian들이 함께 처리되어 cache hit rate 향상
- **Occupancy**: 타일 크기가 warp size(32)의 배수인 256(16×16)으로 GPU 활용 극대화
- **Load Balancing**: 타일마다 처리할 Gaussian 수가 다르지만, 각 타일이 독립적이므로 GPU 스케줄러가 자동 조정

#### 2. Frustum Culling

In [ ]:
// preprocessing 단계에서 카메라 시야 밖의 Gaussian 제거
if (p_view.z < near_plane || p_view.z > far_plane) continue;
if (p_proj.x < 0 || p_proj.x > W) continue;
if (p_proj.y < 0 || p_proj.y > H) continue;


- 렌더링 전에 불필요한 Gaussian을 조기 제거하여 메모리 대역폭 절약
- 타일 할당 단계에서도 bounding box로 추가 필터링

#### 3. Depth Sorting (CUB Radix Sort)

In [ ]:
// GPU에서 고속 정렬 수행
cub::DeviceRadixSort::SortPairs(
    d_temp_storage, temp_storage_bytes,
    gaussian_keys_unsorted, gaussian_keys_sorted,  // (tile_id << 32) | depth
    gaussian_values_unsorted, gaussian_values_sorted,  // Gaussian IDs
    num_rendered, 0, 64  // 64-bit keys
);


- **Tile ID + Depth**: 상위 32비트는 tile ID, 하위 32비트는 depth
- **Front-to-back ordering**: 올바른 alpha blending과 early stopping을 위해 필수
- **O(n log n) → O(n)**: Radix sort는 정수 키에 대해 선형 시간 복잡도

#### 4. Shared Memory Optimization

In [ ]:
// Forward pass: 256 스레드가 협력하여 데이터 로딩
__shared__ int collected_id[BLOCK_SIZE];           // 1 KB
__shared__ float2 collected_xy[BLOCK_SIZE];        // 2 KB
__shared__ float4 collected_conic_opacity[BLOCK_SIZE];  // 4 KB
// Total: ~7 KB per block (48 KB limit 내)


- **Coalesced loading**: 연속된 메모리 접근으로 대역폭 최대화
- **Broadcasting**: 한 번 로딩한 데이터를 256개 픽셀이 재사용
- **Global memory 접근 최소화**: 7-8배 속도 향상

#### 5. Early Stopping

In [ ]:
// 투과율이 임계값 이하로 떨어지면 중단
if (T < 0.0001f) {
    done = true;
    continue;
}


- 대부분의 픽셀은 5-10개의 Gaussian으로 충분
- 불필요한 계산 생략으로 평균 2-3배 속도 향상
- **Warp divergence 최소화**: `__syncthreads_count(done)`로 전체 블록이 완료되면 조기 종료

---

## 사용

In [ ]:
import torch
from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer

# GPU 확인
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

# Rasterizer 생성 테스트
raster_settings = GaussianRasterizationSettings(
    image_height=512,
    image_width=512,
    tanfovx=1.0,
    tanfovy=1.0,
    bg=torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda"),
    scale_modifier=1.0,
    viewmatrix=torch.eye(4, dtype=torch.float32, device="cuda"),
    projmatrix=torch.eye(4, dtype=torch.float32, device="cuda"),
    sh_degree=3,
    campos=torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda"),
    prefiltered=False,
    debug=False
)

rasterizer = GaussianRasterizer(raster_settings=raster_settings)
print("Rasterizer created successfully!")



### 사용 예제

#### Rendering with diff-gaussian-rasterization

```python
import torch
import math
from diff_gaussian_rasterization import GaussianRasterizationSettings, GaussianRasterizer

def render_gaussians(
    means3D,      # [N, 3]
    opacities,    # [N, 1]
    scales,       # [N, 3]
    rotations,    # [N, 4]
    shs,          # [N, 16, 3] for degree 3
    viewmatrix,   # [4, 4]
    projmatrix,   # [4, 4]
    campos,       # [3]
    tanfovx,      # tan(fov_x / 2)
    tanfovy,      # tan(fov_y / 2)
    H=512, W=512
):
    raster_settings = GaussianRasterizationSettings(
        image_height=H,
        image_width=W,
        tanfovx=tanfovx,
        tanfovy=tanfovy,
        bg=torch.tensor([1, 1, 1], dtype=torch.float32, device="cuda"),
        scale_modifier=1.0,
        viewmatrix=viewmatrix,
        projmatrix=projmatrix,
        sh_degree=3,
        campos=campos,
        prefiltered=False,
        debug=False
    )
    
    rasterizer = GaussianRasterizer(raster_settings=raster_settings)
    
    rendered_image, radii = rasterizer(
        means3D=means3D,
        means2D=torch.zeros_like(means3D[:, :2]),  # will be computed
        shs=shs,
        colors_precomp=None,
        opacities=opacities,
        scales=scales,
        rotations=rotations,
        cov3D_precomp=None
    )